In [1]:
# ============================================================
# DEMETER LSTM-AE THRESHOLD SWEEP ANALYSIS
# For already trained and saved rolling-window models
# Thresholds: 95, 96, 97, 98, 99
# Models: A-E and In-A to In-E
# ============================================================

from pathlib import Path
import sys
import os
import random
import json

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader


In [2]:


PROJECT_ROOT = Path(r"C:\PROJECT-DEMETER\Demeter-LSTM_AE")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from lstm import HalfOrbitPairDataset
from lstm.scaling import scale_datasets
from lstm import LSTMAutoencoder
from lstm.anomaly_detection import AnomalyDetector
from lstm import SeismicCriteria
from lstm import SeismicAnalysis


# ============================================================
# 2. Reproducibility
# ============================================================

SEED = 201894

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ============================================================
# 3. Main configuration
# ============================================================

DATA_DIR = PROJECT_ROOT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"

STORM_DATA_PATH = DATA_DIR / "storm_data.pkl"
MAIN_DATA_PATH = DATA_DIR / "Down_Orbits-Q3-loc-Train-limitedgrids_RES.pkl"

EQ_PATH = Path(r"C:\PROJECT-DEMETER\Demeter-LSTM_AE\Data\EQ.csv")

OUTPUT_DIR_RESULTS = PROJECT_ROOT / "outputs"
OUTPUT_DIR_RESULTS.mkdir(parents=True, exist_ok=True)

# Where all trained models are saved
# Expected model name format:
# best_model_Hp_tw48-30dBG_A_w0.pth
# best_model_Hp_tw48-30dBG_In-A_w0.pth
SAVED_MODEL_DIR = Path(r"C:\PROJECT-DEMETER\Demeter-LSTM_AE\Models-Trained\SW22_TW48")

# Output folder for threshold analysis
OUTPUT_THRESHOLD_DIR = OUTPUT_DIR_RESULTS / "Threshold_sweep_SW22_TW48"
OUTPUT_THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)

# Rolling-window setup
train_months = 12
val_months = 3
stride = 3
min_data_points = 17

WINDOW_START = "2005-01-01 00:00:00"
WINDOW_END = "2010-01-02 00:00:00"

# Model common settings
input_size = 11
latent_dim = 2

# Final seismic setting
tw = 48
sw = 22

# Thresholds to test
threshold_values = [95, 98]

# Storm thresholds
Dst = -50
Kp = 3
AE = 500

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Model folder:", SAVED_MODEL_DIR)
print("Output folder:", OUTPUT_THRESHOLD_DIR)



Using device: cpu
Project root: C:\PROJECT-DEMETER\Demeter-LSTM_AE
Data folder: C:\PROJECT-DEMETER\Demeter-LSTM_AE\Data
Model folder: C:\PROJECT-DEMETER\Demeter-LSTM_AE\Models-Trained\SW22_TW48
Output folder: C:\PROJECT-DEMETER\Demeter-LSTM_AE\outputs\Threshold_sweep_SW22_TW48


In [3]:

# 4. Hyperparameter table for all trained models


data = [

    {"Naming": f"tw{tw}-30dBG_A", "Learning rate": 0.001, "Hidden size": 8,  "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_B", "Learning rate": 0.001, "Hidden size": 12, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_C", "Learning rate": 0.001, "Hidden size": 18, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_D", "Learning rate": 0.001, "Hidden size": 32, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_E", "Learning rate": 0.001, "Hidden size": 64, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},

    {"Naming": f"tw{tw}-30dBG_In-A", "Learning rate": 0.001, "Hidden size": 8,  "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_In-B", "Learning rate": 0.001, "Hidden size": 12, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_In-C", "Learning rate": 0.001, "Hidden size": 18, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_In-D", "Learning rate": 0.001, "Hidden size": 32, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},
    {"Naming": f"tw{tw}-30dBG_In-E", "Learning rate": 0.001, "Hidden size": 64, "Number of Layers": 2, "Batch size": 8, "tw": tw, "sw": sw},

]

df_hparams = pd.DataFrame(data)
display(df_hparams)



,Naming,Learning rate,Hidden size,Number of Layers,Batch size,tw,sw
0,tw48-30dBG_A,0.001,8,2,8,48,22
1,tw48-30dBG_B,0.001,12,2,8,48,22
2,tw48-30dBG_C,0.001,18,2,8,48,22
3,tw48-30dBG_D,0.001,32,2,8,48,22
4,tw48-30dBG_E,0.001,64,2,8,48,22
5,tw48-30dBG_In-A,0.001,8,2,8,48,22
6,tw48-30dBG_In-B,0.001,12,2,8,48,22
7,tw48-30dBG_In-C,0.001,18,2,8,48,22
8,tw48-30dBG_In-D,0.001,32,2,8,48,22
9,tw48-30dBG_In-E,0.001,64,2,8,48,22


In [4]:

# ============================================================
# 5. Load DEMETER data, earthquake catalogue, and storm data
# ============================================================

storm_data = pd.read_pickle(STORM_DATA_PATH)
data_main = pd.read_pickle(MAIN_DATA_PATH)

df = data_main.loc[:, ~data_main.columns.str.startswith("Q3")].copy()
df.index = pd.to_datetime(df.index)

eq = pd.read_csv(EQ_PATH, parse_dates=["Time"])
eq["Time"] = pd.to_datetime(eq["Time"], errors="coerce")
eq = eq.dropna(subset=["Time"]).copy()
eq["Time"] = eq["Time"].dt.tz_localize(None)

storm_data["Datetime"] = pd.to_datetime(storm_data["Datetime"], errors="coerce")

storm_all = storm_data[
    (storm_data["Dst"] < Dst) |
    (storm_data["Kp"] > Kp) |
    (storm_data["AE"] > AE)
].dropna(subset=["Datetime"]).copy()

print("DEMETER dataframe shape:", df.shape)
print("Earthquake catalogue shape:", eq.shape)
print("Storm-filtered rows:", storm_all.shape)



DEMETER dataframe shape: (456539, 13)
Earthquake catalogue shape: (12442, 5)
Storm-filtered rows: (6318, 4)


In [5]:

# ============================================================
# 6. Helper functions
# ============================================================

def generate_windows(start_date, end_date, train_months, val_months, stride):
    windows = []
    current = pd.to_datetime(start_date)
    final = pd.to_datetime(end_date)

    while current + pd.DateOffset(months=train_months + val_months) < final:
        train_start = current
        train_end = current + pd.DateOffset(months=train_months)
        val_start = train_end
        val_end = train_end + pd.DateOffset(months=val_months)

        windows.append({
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
        })

        current = current + pd.DateOffset(months=stride)

    return windows


def correct_anomalies_for_storms(datetime_sequences, anomalies_agg, anomalies_fb, storm_all):
    storm_data_indices = []
    storm_times = pd.to_datetime(storm_all["Datetime"], errors="coerce").dropna()

    for idx, seq in enumerate(datetime_sequences):
        seq_times = pd.to_datetime(seq)
        start_t = seq_times[0]
        end_t = seq_times[-1]

        if storm_times.between(start_t, end_t).any():
            storm_data_indices.append(idx)

    corrected_anomalies_agg = [
        idx for idx in anomalies_agg
        if idx not in storm_data_indices
    ]

    corrected_anomalies_fb = [[] for _ in range(len(anomalies_fb))]

    for feature_index, anomalies in enumerate(anomalies_fb):
        corrected_anomalies_fb[feature_index] = [
            idx for idx in anomalies
            if idx not in storm_data_indices
        ]

    return corrected_anomalies_agg, corrected_anomalies_fb, storm_data_indices


def run_seismic_analysis(
    dataset,
    test_dataset,
    eq,
    seismic_criteria,
    corrected_test_fb_anomalies,
    corrected_test_anomalies,
    model_name,
    output_dir,
    data_label="",
    threshold_label="",
):
    sa = SeismicAnalysis(
        dataset=dataset,
        earthquake_catalog=eq,
        create_half_orbit_sequences=test_dataset.create_half_orbit_sequences,
        is_eq_fn=seismic_criteria.is_eq,
        model_name=f"{model_name}",
        threshold_label=str(threshold_label),
        mean_rst=0,
        sigma_rst=0,
        output_dir=output_dir,
    )

    seismic_seqs_agg, matched_eqs_agg, missed_eqs_agg = sa.agg_analysis(
        data_label=data_label,
        anomalous_indices=corrected_test_anomalies,
        plot=False,
    )

    total_sequences_agg = len(corrected_test_anomalies)
    seismic_count = len(seismic_seqs_agg)
    total_eq = len(matched_eqs_agg)

    if total_sequences_agg > 0:
        if seismic_count == total_sequences_agg:
            p_agg = (seismic_count - 1) / total_sequences_agg
        elif seismic_count == 0:
            p_agg = 1 / total_sequences_agg
        else:
            p_agg = seismic_count / total_sequences_agg

        agg_value = p_agg * 100
        agg_error = np.sqrt(p_agg * (1 - p_agg) / total_sequences_agg) * 100
    else:
        agg_value = 0
        agg_error = 0

    return {
        "agg": {
            "value": agg_value,
            "error": agg_error,
            "total_eq": total_eq,
            "seismic_indices": seismic_seqs_agg,
        }
    }


def json_safe(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, list):
        return [json_safe(x) for x in obj]
    if isinstance(obj, tuple):
        return [json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    return obj


def to_json_string(obj):
    return json.dumps(json_safe(obj), ensure_ascii=False)


In [6]:


# ============================================================
# 7. Generate rolling windows
# ============================================================

windows_train = generate_windows(
    start_date=WINDOW_START,
    end_date=WINDOW_END,
    train_months=train_months,
    val_months=val_months,
    stride=stride,
)

print("Number of rolling windows:", len(windows_train))

for i, w in enumerate(windows_train):
    print(
        f"Window {i}: "
        f"Train {w['train_start']} -> {w['train_end']} | "
        f"Val {w['val_start']} -> {w['val_end']}"
    )



Number of rolling windows: 16
Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
Window 1: Train 2005-04-01 00:00:00 -> 2006-04-01 00:00:00 | Val 2006-04-01 00:00:00 -> 2006-07-01 00:00:00
Window 2: Train 2005-07-01 00:00:00 -> 2006-07-01 00:00:00 | Val 2006-07-01 00:00:00 -> 2006-10-01 00:00:00
Window 3: Train 2005-10-01 00:00:00 -> 2006-10-01 00:00:00 | Val 2006-10-01 00:00:00 -> 2007-01-01 00:00:00
Window 4: Train 2006-01-01 00:00:00 -> 2007-01-01 00:00:00 | Val 2007-01-01 00:00:00 -> 2007-04-01 00:00:00
Window 5: Train 2006-04-01 00:00:00 -> 2007-04-01 00:00:00 | Val 2007-04-01 00:00:00 -> 2007-07-01 00:00:00
Window 6: Train 2006-07-01 00:00:00 -> 2007-07-01 00:00:00 | Val 2007-07-01 00:00:00 -> 2007-10-01 00:00:00
Window 7: Train 2006-10-01 00:00:00 -> 2007-10-01 00:00:00 | Val 2007-10-01 00:00:00 -> 2008-01-01 00:00:00
Window 8: Train 2007-01-01 00:00:00 -> 2008-01-01 00:00:00 | Val 2008-01-01 00:00:00 -> 2008-04-01 00:00:0

In [7]:

# ============================================================
# 8. Main threshold-sweep loop
# ============================================================

all_results = []

for _, hp in df_hparams.iterrows():

    model_name = hp["Naming"]
    lr = float(hp["Learning rate"])
    hidden_size = int(hp["Hidden size"])
    num_layers = int(hp["Number of Layers"])
    batch_size = int(hp["Batch size"])
    model_tw = int(hp["tw"])
    model_sw = int(hp["sw"])

    seismic_criteria = SeismicCriteria(
        spatial_width=model_sw,
        time_window_hours=model_tw,
    )

    print("\n" + "#" * 100)
    print(f"MODEL: {model_name}")
    print(
        f"lr={lr}, hidden_size={hidden_size}, num_layers={num_layers}, "
        f"batch_size={batch_size}, TW={model_tw}, SW={model_sw}"
    )
    print("#" * 100)

    for i, w in enumerate(windows_train):

        tag_l = f"tw{model_tw}_w{i}"
        tag = f"Hp_{model_name}_w{i}"

        print("\n" + "=" * 90)
        print(
            f"Model {model_name} | Window {i}: "
            f"Train {w['train_start']} -> {w['train_end']} | "
            f"Val {w['val_start']} -> {w['val_end']}"
        )
        print("=" * 90)

        # ------------------------------------------------------------
        # Load background-corrected data
        # ------------------------------------------------------------
        bg_window_path = BG_WINDOW_DIR / f"Background_data-window_{i}.pkl"

        if not bg_window_path.exists():
            raise FileNotFoundError(f"Missing background window file: {bg_window_path}")

        dfw = pd.read_pickle(bg_window_path)
        dfw = dfw.loc[:, ~dfw.columns.str.startswith("Q3")].copy()
        dfw.index = pd.to_datetime(dfw.index)

        train_set = dfw[
            (dfw.index >= w["train_start"]) &
            (dfw.index < w["train_end"])
        ]

        val_set = dfw[
            (dfw.index >= w["val_start"]) &
            (dfw.index < w["val_end"])
        ]

        test_set = pd.DataFrame(df[df.index >= "2010-01-01"])

        # ------------------------------------------------------------
        # Build datasets
        # ------------------------------------------------------------
        train_dataset = HalfOrbitPairDataset(
            train_set,
            min_data_points=min_data_points,
        )

        val_dataset = HalfOrbitPairDataset(
            val_set,
            min_data_points=min_data_points,
        )

        test_dataset = HalfOrbitPairDataset(
            test_set,
            min_data_points=min_data_points,
        )

        # ------------------------------------------------------------
        # Load seismic labels for normal-only scaler fitting
        # ------------------------------------------------------------
        train_label_path = LABEL_DIR / f"summary_df_train_30D-22SW-{tag_l}.csv"
        val_label_path = LABEL_DIR / f"summary_df_val_30D-22SW-{tag_l}.csv"

        if not train_label_path.exists():
            raise FileNotFoundError(f"Missing train label file: {train_label_path}")

        if not val_label_path.exists():
            raise FileNotFoundError(f"Missing validation label file: {val_label_path}")

        df_train_labels = pd.read_csv(train_label_path)
        df_val_labels = pd.read_csv(val_label_path)

        train_dataset_normal = HalfOrbitPairDataset(
            train_set,
            df_train_labels,
            min_data_points,
            use_label_0_only=True,
        )

        val_dataset_normal = HalfOrbitPairDataset(
            val_set,
            df_val_labels,
            min_data_points,
            use_label_0_only=True,
        )

        # ------------------------------------------------------------
        # Fit scaler using normal train set, then transform full train/val
        # ------------------------------------------------------------
        _scaled_train_data_n, _scaled_val_data_n, _scaled_test_data, _mean = scale_datasets(
            train_dataset_normal,
            train_dataset,
            val_dataset,
            test_dataset,
            fit=True,
        )

        scaled_train_data, scaled_val_data, _, _ = scale_datasets(
            train_dataset_normal,
            train_dataset,
            val_dataset,
            test_dataset,
            fit=False,
        )

        train_data_loader = DataLoader(
            scaled_train_data,
            batch_size=1,
            shuffle=False,
        )

        val_data_loader = DataLoader(
            scaled_val_data,
            batch_size=1,
            shuffle=False,
        )

        print("Train sequences:", len(scaled_train_data))
        print("Validation sequences:", len(scaled_val_data))

        # ------------------------------------------------------------
        # Load trained model
        # ------------------------------------------------------------
        model = LSTMAutoencoder(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            latent_dim=latent_dim,
        )

        best_model_path = SAVED_MODEL_DIR / f"best_model_{tag}.pth"

        if not best_model_path.exists():
            raise FileNotFoundError(f"Missing saved model: {best_model_path}")

        model.load_state_dict(
            torch.load(best_model_path, map_location=DEVICE)
        )

        model.to(DEVICE)
        model.eval()

        print(f"[OK] Loaded model: {best_model_path}")

        # ------------------------------------------------------------
        # Compute reconstruction errors ONCE per model-window
        # ------------------------------------------------------------
        train_detector = AnomalyDetector(
            model,
            dataloader=train_data_loader,
            num_features=input_size,
        )

        val_detector = AnomalyDetector(
            model,
            dataloader=val_data_loader,
            num_features=input_size,
        )

        train_errors_agg = train_detector.compute_reconstruction_errors_agg(
            train_data_loader
        )

        train_errors_fb = train_detector.compute_reconstruction_errors_fb(
            train_data_loader
        )

        val_errors_agg = val_detector.compute_reconstruction_errors_agg(
            val_data_loader
        )

        val_errors_fb = val_detector.compute_reconstruction_errors_fb(
            val_data_loader
        )

        # ------------------------------------------------------------
        # Create half-orbit sequences once for storm correction
        # ------------------------------------------------------------
        train_sequences, train_datetime_sequences, train_lat_long_sequences, _ = (
            train_dataset.create_half_orbit_sequences(train_set)
        )

        val_sequences, val_datetime_sequences, val_lat_long_sequences, _ = (
            val_dataset.create_half_orbit_sequences(val_set)
        )

        # ------------------------------------------------------------
        # Loop over thresholds: 95 to 99
        # ------------------------------------------------------------
        for pc_i in threshold_values:

            print(f"Running threshold percentile: {pc_i}")

            # Threshold estimated from TRAIN reconstruction errors
            threshold_agg = np.percentile(train_errors_agg, pc_i)
            threshold_fb = np.percentile(train_errors_fb, pc_i, axis=0)

            # Apply threshold to train
            train_anomalies_agg = train_detector.detect_anomalies_agg(
                train_errors_agg,
                threshold_agg,
            )

            train_anomalies_fb = train_detector.detect_anomalies_fb(
                train_errors_fb,
                threshold_fb,
            )

            # Apply same train-derived threshold to validation
            val_anomalies_agg = val_detector.detect_anomalies_agg(
                val_errors_agg,
                threshold_agg,
            )

            val_anomalies_fb = val_detector.detect_anomalies_fb(
                val_errors_fb,
                threshold_fb,
            )

            # ------------------------------------------------------------
            # Storm correction
            # ------------------------------------------------------------
            corrected_train_anomalies_agg, corrected_train_anomalies_fb, storm_train_data_indices = (
                correct_anomalies_for_storms(
                    train_datetime_sequences,
                    train_anomalies_agg,
                    train_anomalies_fb,
                    storm_all,
                )
            )

            corrected_val_anomalies_agg, corrected_val_anomalies_fb, storm_val_data_indices = (
                correct_anomalies_for_storms(
                    val_datetime_sequences,
                    val_anomalies_agg,
                    val_anomalies_fb,
                    storm_all,
                )
            )

            train_anomalies_before = len(train_anomalies_agg)
            train_anomalies_after = len(corrected_train_anomalies_agg)

            val_anomalies_before = len(val_anomalies_agg)
            val_anomalies_after = len(corrected_val_anomalies_agg)

            # ------------------------------------------------------------
            # Seismic analysis before storm correction
            # ------------------------------------------------------------
            results_train = run_seismic_analysis(
                train_set,
                train_dataset,
                eq,
                seismic_criteria,
                train_anomalies_fb,
                train_anomalies_agg,
                model_name=f"EQ_{tag}_pc{pc_i}",
                output_dir=str(OUTPUT_THRESHOLD_DIR),
                data_label=f"Train_{tag_l}_pc{pc_i}",
                threshold_label=pc_i,
            )

            results_val = run_seismic_analysis(
                val_set,
                val_dataset,
                eq,
                seismic_criteria,
                val_anomalies_fb,
                val_anomalies_agg,
                model_name=f"EQ_{tag}_pc{pc_i}",
                output_dir=str(OUTPUT_THRESHOLD_DIR),
                data_label=f"Val_{tag_l}_pc{pc_i}",
                threshold_label=pc_i,
            )

            # ------------------------------------------------------------
            # Seismic analysis after storm correction
            # ------------------------------------------------------------
            results_train_sc = run_seismic_analysis(
                train_set,
                train_dataset,
                eq,
                seismic_criteria,
                corrected_train_anomalies_fb,
                corrected_train_anomalies_agg,
                model_name=f"stormC_EQ_{tag}_pc{pc_i}",
                output_dir=str(OUTPUT_THRESHOLD_DIR),
                data_label=f"Train_{tag_l}_pc{pc_i}",
                threshold_label=pc_i,
            )

            results_val_sc = run_seismic_analysis(
                val_set,
                val_dataset,
                eq,
                seismic_criteria,
                corrected_val_anomalies_fb,
                corrected_val_anomalies_agg,
                model_name=f"stormC_EQ_{tag}_pc{pc_i}",
                output_dir=str(OUTPUT_THRESHOLD_DIR),
                data_label=f"Val_{tag_l}_pc{pc_i}",
                threshold_label=pc_i,
            )

            # ------------------------------------------------------------
            # Store TRAIN result
            # ------------------------------------------------------------
            all_results.append({
                "model_name": model_name,
                "threshold_percentile": pc_i,
                "window": i,
                "split": "Train",

                "learning_rate": lr,
                "hidden_size": hidden_size,
                "num_layers": num_layers,
                "batch_size": batch_size,
                "tw": model_tw,
                "sw": model_sw,

                "train_start": w["train_start"],
                "train_end": w["train_end"],
                "val_start": w["val_start"],
                "val_end": w["val_end"],

                "threshold_agg": float(threshold_agg),
                "threshold_fb": threshold_fb.tolist(),

                "anomalies": train_anomalies_before,
                "anomalies_sc": train_anomalies_after,
                "storm_removed_anomalies": train_anomalies_before - train_anomalies_after,
                "storm_sequence_count": len(storm_train_data_indices),

                "agg_value": results_train["agg"]["value"],
                "agg_error": results_train["agg"]["error"],
                "agg_value_sc": results_train_sc["agg"]["value"],
                "agg_error_sc": results_train_sc["agg"]["error"],

                "agg_total_eq": results_train["agg"]["total_eq"],
                "agg_total_eq_sc": results_train_sc["agg"]["total_eq"],

                "seismic_indices": results_train["agg"]["seismic_indices"],
                "seismic_indices_sc": results_train_sc["agg"]["seismic_indices"],

                "model_path": str(best_model_path),
            })

            # ------------------------------------------------------------
            # Store VALIDATION result
            # ------------------------------------------------------------
            all_results.append({
                "model_name": model_name,
                "threshold_percentile": pc_i,
                "window": i,
                "split": "Val",

                "learning_rate": lr,
                "hidden_size": hidden_size,
                "num_layers": num_layers,
                "batch_size": batch_size,
                "tw": model_tw,
                "sw": model_sw,

                "train_start": w["train_start"],
                "train_end": w["train_end"],
                "val_start": w["val_start"],
                "val_end": w["val_end"],

                "threshold_agg": float(threshold_agg),
                "threshold_fb": threshold_fb.tolist(),

                "anomalies": val_anomalies_before,
                "anomalies_sc": val_anomalies_after,
                "storm_removed_anomalies": val_anomalies_before - val_anomalies_after,
                "storm_sequence_count": len(storm_val_data_indices),

                "agg_value": results_val["agg"]["value"],
                "agg_error": results_val["agg"]["error"],
                "agg_value_sc": results_val_sc["agg"]["value"],
                "agg_error_sc": results_val_sc["agg"]["error"],

                "agg_total_eq": results_val["agg"]["total_eq"],
                "agg_total_eq_sc": results_val_sc["agg"]["total_eq"],

                "seismic_indices": results_val["agg"]["seismic_indices"],
                "seismic_indices_sc": results_val_sc["agg"]["seismic_indices"],

                "model_path": str(best_model_path),
            })

        # ------------------------------------------------------------
        # Save interim CSV after each model-window
        # ------------------------------------------------------------
        interim_df = pd.DataFrame(all_results)

        interim_save = interim_df.copy()
        interim_save["threshold_fb"] = interim_save["threshold_fb"].apply(to_json_string)
        interim_save["seismic_indices"] = interim_save["seismic_indices"].apply(to_json_string)
        interim_save["seismic_indices_sc"] = interim_save["seismic_indices_sc"].apply(to_json_string)

        interim_csv = OUTPUT_THRESHOLD_DIR / "threshold_sweep_ALL_MODELS_INTERIM.csv"
        interim_save.to_csv(interim_csv, index=False)

        print(f"[OK] Saved interim result: {interim_csv}")



####################################################################################################
MODEL: tw48-30dBG_A
lr=0.001, hidden_size=8, num_layers=2, batch_size=8, TW=48, SW=22
####################################################################################################

Model tw48-30dBG_A | Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
Train sequences: 1553
Validation sequences: 381
[OK] Loaded model: C:\PROJECT-DEMETER\Demeter-LSTM_AE\Models-Trained\SW22_TW48\best_model_Hp_tw48-30dBG_A_w0.pth
[INFO] Total valid combined sequences: 1553
[INFO] Total valid combined sequences: 381
Running threshold percentile: 95
[INFO] Total valid combined sequences: 1553
[INFO] Total valid combined sequences: 381
[INFO] Total valid combined sequences: 1553
[INFO] Total valid combined sequences: 381
Running threshold percentile: 98
[INFO] Total valid combined sequences: 1553
[INFO] Total valid combined sequences: 381
[INFO]

PermissionError: [Errno 13] Permission denied: 'C:\\PROJECT-DEMETER\\Demeter-LSTM_AE\\outputs\\Threshold_sweep_SW22_TW48\\threshold_sweep_ALL_MODELS_INTERIM.csv'

In [ ]:


# ============================================================
# 9. Save final CSV and pickle
# ============================================================

df_threshold_results = pd.DataFrame(all_results)

final_csv_df = df_threshold_results.copy()
final_csv_df["threshold_fb"] = final_csv_df["threshold_fb"].apply(to_json_string)
final_csv_df["seismic_indices"] = final_csv_df["seismic_indices"].apply(to_json_string)
final_csv_df["seismic_indices_sc"] = final_csv_df["seismic_indices_sc"].apply(to_json_string)

final_csv = OUTPUT_THRESHOLD_DIR / "threshold_sweep_ALL_MODELS_SW22_TW48_pc95_to_99.csv"
final_pkl = OUTPUT_THRESHOLD_DIR / "threshold_sweep_ALL_MODELS_SW22_TW48_pc95_to_99.pkl"

final_csv_df.to_csv(final_csv, index=False)
df_threshold_results.to_pickle(final_pkl)

print("\nDONE")
print("Final CSV saved to:", final_csv)
print("Final pickle saved to:", final_pkl)

print("Expected number of rows:")
print(f"{len(df_hparams)} models × {len(windows_train)} windows × {len(threshold_values)} thresholds × 2 splits")
print("Actual rows:", len(df_threshold_results))

display(df_threshold_results)